# S6E10 — 08 TabPFN members from the public dataset

> **Pipeline position:** 07 RealMLP → **08 TabPFN import** → 09 rich features → 10 → 11 stack

| | |
|---|---|
| **Input** | `data/external/s6e10-tabpfn-member/` (`meta.json`, `oof_*.csv`, `test_*.csv`) |
| **Output** | `preds/oof_tabpfn_avg.npy`, `oof_tabpfn_raw_avg.npy`, `oof_tabicl.npy` (+ `test_*`) |
| **Runtime** | seconds |
| **Result** | `tabpfn_avg` OOF AUC **0.96158** — the best single model in the project, and the top weight in the stack |

**What TabPFN is:** a transformer pre-trained on millions of synthetic tables. It classifies a row by reading labelled rows as *context* in a single forward pass — no gradient training on our data. A full 5-fold run with the whole 560k-row context needs a machine far bigger than ours, so we use the predictions goodpjw2008 published as the Kaggle dataset **`goodpjw2008/s6e10-tabpfn-member`** (credit to the author and to hemingweb / Prior Labs for the idea and the model). They are out-of-fold on the same split as ours (`StratifiedKFold(5, shuffle=True, random_state=42)`).

**Before any use we verify the alignment:** OOF AUC and per-fold AUCs, recomputed on *our* labels and folds, must equal the numbers in `meta.json`. Otherwise rows are misaligned and the member must not enter the stack.

**How to get the data:** open https://www.kaggle.com/datasets/goodpjw2008/s6e10-tabpfn-member → *Download* → unzip into `data/external/s6e10-tabpfn-member/` (must contain `meta.json` and `oof_*.csv` / `test_*.csv`). On Kaggle: attach it via *Add Input*.

> **Glossary:** *In-context learning* = the model "learns" from examples given in its input, without weight updates. *Member* = one prediction set (one configuration / seed) from the dataset.

In [1]:
import json, glob
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

N_FOLDS, RANDOM_STATE = 5, 42
RAW = next(p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')] if (p / 'y_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds')); PREDS_DIR.mkdir(exist_ok=True)

# Search every place the dataset may be: local data/external, ../data/external, or attached on Kaggle.
meta_hits = sorted(glob.glob('data/external/**/meta.json', recursive=True) + glob.glob('../data/external/**/meta.json', recursive=True)
                   + glob.glob('/kaggle/input/**/s6e10-tabpfn-member/**/meta.json', recursive=True) + glob.glob('/kaggle/input/**/meta.json', recursive=True))
assert meta_hits, 'meta.json not found: download the dataset into data/external/s6e10-tabpfn-member/ (see the text above)'
DS = Path(meta_hits[0]).parent
meta = json.load(open(DS / 'meta.json'))
print('dataset folder:', DS)
print('members in meta.json:', list(meta['members']))

train_ids = pd.read_csv(RAW / 'train.csv', usecols=['id'])['id'].values
test_ids = pd.read_csv(RAW / 'test.csv', usecols=['id'])['id'].values
y = pd.read_parquet(PROC / 'y_train.parquet')['y'].astype(int).values
assert len(train_ids) == len(y) == 699635 and len(test_ids) == 299844
# Our folds. The dataset author used the same split; section 1 verifies it.
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(np.zeros(len(y)), y))

dataset folder: data/external/s6e10-tabpfn-member
members in meta.json: ['tabpfn_full_raw', 'tabpfn_full_raw_s1', 'tabpfn_te', 'tabpfn_te4', 'tabpfn_catfd', 'tabpfn_catfd_s1', 'tabpfn_catfd10', 'tabicl_full_raw']


## 1. Load and verify every member

In [2]:
OOF, TEST, rows = {}, {}, []
for name in meta['members']:
    fo, ft = DS / f'oof_{name}.csv', DS / f'test_{name}.csv'
    if not (fo.exists() and ft.exists()): print('skipped (files missing):', name); continue
    o = pd.read_csv(fo).set_index('id')['pred']; t = pd.read_csv(ft).set_index('id')['pred']
    # Align by id, not by row order: the CSVs may be sorted differently.
    o, t = o.reindex(train_ids).values, t.reindex(test_ids).values
    assert not np.isnan(o).any() and not np.isnan(t).any(), f'{name}: ids do not cover our train/test'
    ours_fold = [roc_auc_score(y[va], o[va]) for _, va in FOLDS]
    theirs_fold = meta['members'][name]['fold_auc']
    OOF[name], TEST[name] = o, t
    rows.append({'member': name, 'OOF AUC (ours)': round(roc_auc_score(y, o), 5), 'OOF AUC (meta)': round(meta['members'][name]['oof_auc'], 5),
                 'max |fold diff|': round(max(abs(a - b) for a, b in zip(ours_fold, theirs_fold)), 6)})
tab = pd.DataFrame(rows); print(tab.to_string(index=False))
# Safety gate: a member whose per-fold AUCs differ from meta.json by more than 2e-4 is NOT used (misaligned rows would silently leak or destroy the stack).
bad = tab[tab['max |fold diff|'] > 2e-4]
print('\nALIGNMENT OK: every member matches its meta.json on our labels and folds' if bad.empty else f'\nWARNING: not aligned: {bad.member.tolist()} -> do not use these members')

            member  OOF AUC (ours)  OOF AUC (meta)  max |fold diff|
   tabpfn_full_raw         0.96073         0.96073              0.0
tabpfn_full_raw_s1         0.96063         0.96063              0.0
         tabpfn_te         0.96110         0.96110              0.0
        tabpfn_te4         0.96114         0.96114              0.0
      tabpfn_catfd         0.96092         0.96092              0.0
   tabpfn_catfd_s1         0.96089         0.96089              0.0
    tabpfn_catfd10         0.96107         0.96107              0.0
   tabicl_full_raw         0.95759         0.95759              0.0

ALIGNMENT OK: every member matches its meta.json on our labels and folds


## 2. Average the route-aware members and save
Five route-aware members (`tabpfn_te`, `tabpfn_te4`, `tabpfn_catfd`, `tabpfn_catfd_s1`, `tabpfn_catfd10`) are averaged on the logit scale into one member `tabpfn_avg` — they are very similar to each other, so one averaged member serves the stack better than five near-duplicates. Raw-column members → `tabpfn_raw_avg`; TabICL v2 → `tabicl`. Both are weaker but offer a different view; `11_stack` uses them as separate members.

In [3]:
lg = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
sg = lambda z: 1 / (1 + np.exp(-z))
# Route-aware members: TabPFN was given route features (target-encoded / categorical flight_distance).
ROUTE = [m for m in ['tabpfn_te', 'tabpfn_te4', 'tabpfn_catfd', 'tabpfn_catfd_s1', 'tabpfn_catfd10'] if m in OOF and m not in (bad.member.tolist() if not bad.empty else [])]
RAWM = [m for m in ['tabpfn_full_raw', 'tabpfn_full_raw_s1'] if m in OOF and m not in (bad.member.tolist() if not bad.empty else [])]
# Average on the logit scale, save as probabilities (same convention as every other model).
def save_avg(members, tag):
    o = np.mean([lg(OOF[m]) for m in members], axis=0); t = np.mean([lg(TEST[m]) for m in members], axis=0)
    np.save(PREDS_DIR / f'oof_{tag}.npy', sg(o)); np.save(PREDS_DIR / f'test_{tag}.npy', sg(t))
    print(f'{tag}: {len(members)} members, OOF AUC {roc_auc_score(y, o):.5f}, per fold', [round(roc_auc_score(y[va], o[va]), 5) for _, va in FOLDS], '-> saved')
    return o
o_route = save_avg(ROUTE, 'tabpfn_avg') if ROUTE else None
o_raw = save_avg(RAWM, 'tabpfn_raw_avg') if RAWM else None
o_icl = save_avg(['tabicl_full_raw'], 'tabicl') if 'tabicl_full_raw' in OOF and 'tabicl_full_raw' not in (bad.member.tolist() if not bad.empty else []) else None

# correlation with our own models (if their OOF files are in preds/)
from scipy.stats import spearmanr
for nm in ['realmlp', 'xgb', 'cat_v1', 'mlp']:
    f = PREDS_DIR / f'oof_{nm}.npy'
    if f.exists() and o_route is not None: print(f'Spearman tabpfn_avg vs {nm}: {spearmanr(o_route, np.load(f))[0]:.4f}')

tabpfn_avg: 5 members, OOF AUC 0.96158, per fold [0.96167, 0.96081, 0.96212, 0.96183, 0.96149] -> saved
tabpfn_raw_avg: 2 members, OOF AUC 0.96089, per fold [0.96082, 0.9601, 0.96148, 0.96122, 0.96086] -> saved
tabicl: 1 members, OOF AUC 0.95759, per fold [0.9574, 0.95693, 0.95813, 0.95785, 0.95766] -> saved
Spearman tabpfn_avg vs realmlp: 0.9720
Spearman tabpfn_avg vs xgb: 0.9690
Spearman tabpfn_avg vs cat_v1: 0.9705
Spearman tabpfn_avg vs mlp: 0.9678


## Results & decisions

| Member | OOF AUC | Aligned with `meta.json` |
|---|---|---|
| `tabpfn_te4` | 0.96114 | ✔ |
| `tabpfn_te` | 0.96110 | ✔ |
| `tabpfn_catfd10` | 0.96107 | ✔ |
| `tabpfn_catfd` / `_s1` | 0.96092 / 0.96089 | ✔ |
| `tabpfn_full_raw` / `_s1` | 0.96073 / 0.96063 | ✔ |
| `tabicl_full_raw` | 0.95759 | ✔ |
| **`tabpfn_avg` (5 route-aware, logit mean)** | **0.96158** | |
| `tabpfn_raw_avg` (2 raw) | 0.96089 | |

- **Route-aware inputs help TabPFN** (0.96158 vs 0.96089 for raw columns): the same route insight that worked for GBDTs and for RealMLP.
- **Averaging near-duplicates still pays**: single members are 0.9609–0.9611, the average 0.96158.
- **TabPFN is a *different* model**: Spearman correlation with our own models is only 0.968–0.972 (RealMLP 0.9720, XGB 0.9690, CatBoost 0.9705, MLP 0.9678). Lower correlation = more complementary — which is why it receives the largest weight in the stack.
- **Limitation:** these are someone else's predictions; we cannot re-run them with new features. Fine for the stack, not a basis for our own experiments.

## Next steps
Rerun `11_stack.ipynb`: it already looks for `oof_tabpfn_avg.npy` and uses it as the member `tabpfn`. Decide by the nested-CV table (gain over the stack without it, per fold), not by the leaderboard.